# testing

> Utilities for testing processing and statistics.

In [ ]:
# | default_exp testing

In [ ]:
# | export

import numpy as np
from astropy.convolution import convolve, TrapezoidDisk2DKernel
from FyeldGenerator import generate_field
from scipy.stats import norm

In [ ]:
# | export


def _distrib(shape):
    a = np.random.normal(loc=0, scale=1, size=shape)
    b = np.random.normal(loc=0, scale=1, size=shape)
    return a + 1j * b


def create_test_background(
    shape=(2040, 1020),  # shape of the simulated image,
    rms=1.0,  # rms of the Gaussian background noise, in absence of spatial variations
    background_scale=100,  # scale, in pixels, of the spatial background variations
    background_rms=0.1,  # rms of the spatial background variations
):  # `random`, `background`, `random_with_background`
    """Create a background image for testing.

    A `random` flat background is created by sampling a Gaussian distribution with the
    specified `rms`. A spatially varying `background` is created as a Gaussian random field,
    with a Gaussian power spectrum, centred at zero wavelength with a standard deviation of
    0.1 / `background_scale`. This produces smooth spatial features on scales on the order of
    `background_scale`. This spatially varying background is scaled to have an rms of
    `background_rms`. The two are summed to produce `random_with_background`.
    """
    random = norm(0, rms).rvs(size=shape)
    if (
        background_rms is not None
        and background_rms > 0
        and background_scale is not None
        and background_scale > 0
    ):
        background = generate_field(
            _distrib, norm(0, 0.1 / background_scale).pdf, shape
        )
        background *= background_rms / background.std()
    else:
        background = np.zeros(shape)
    random_with_background = random + background
    return random, background, random_with_background


def create_test_mask(
    shape=(2040, 1020),  # shape of the simulated image,
    scale=100,  # scale, in pixels, of the spatial variations used to create the mask
    threshold=1.5,  # threshold of the spatial variations, as a multiple of the rms
):  # boolean `mask`, in which True means a pixel should be masked
    """Create an arbitrary mask for testing, by thresholding a Guassian random field."""
    mask = generate_field(_distrib, norm(0, 1 / scale).pdf, shape)
    mask = mask > threshold * mask.std()
    return mask


def correlate_pixels(img, slope=0.8, verbose=True):
    kernel = TrapezoidDisk2DKernel(0, slope)
    n = kernel.shape[0]
    f_centre = kernel.array[(n - 1) // 2, (n - 1) // 2]
    if verbose:
        print(f"Fraction of flux in central pixel is {100 * f_centre:.0f}%")
    convolved_img = convolve(img, kernel)
    return convolved_img